# Proyecto E-commerce con Análisis RFM

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt
from pathlib import Path
import zipfile

In [8]:
# checamos que la libreta data tenga mi archivo (se llama archive_retail.zip)
DATA_DIR = Path("data/E-commerse")
Zip_Path = DATA_DIR/"archive_retail.zip"

DATA_DIR.mkdir(parents=True,exist_ok = True)
if not Zip_Path.exists():
    #Si no existe descargamos y almacenamos en la ruta
    !kaggle datasets download -d vijayuv/onlineretail -P {DATA_DIR}

with zipfile.ZipFile(Zip_Path,'r') as zip_ref:
    zip_ref.extractall(DATA_DIR)

print("Dataset descargado y descomprimido")

Dataset descargado y descomprimido


# Leemos el archivo con pandas

In [14]:
#Archivo Path
Archivo_Path = Path(r'.\data\E-commerse\OnlineRetail.csv')
df = pd.read_csv(Archivo_Path,encoding='ISO-8859-1')
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


# Resumen del archivo

In [15]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 33.1 MB


# Limpieza del dataset

In [21]:
#Eliminamos columnas de CustomerID nulas
df = df.dropna(subset=['CustomerID'])
#Filtramos columna Quantity y unitPrice mayores a cero, eliiminando cancelaciones y posibles errores
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
#Convertir InvoiceDate  a objeto datetime de pandas
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
#Crear columna de Gasto Total por linea
df['TotalSum'] = df['Quantity'] * df['UnitPrice']

print(f"Registros de la limpieza: {df.shape[0]}")
df.describe()

Registros de la limpieza: 397884


,Quantity,InvoiceDate,UnitPrice,CustomerID,TotalSum
count,397884.000000,397884,397884.000000,397884.000000,397884.000000
mean,12.988238,2011-07-10 23:41:23.511023,3.116488,15294.423453,22.397000
min,1.000000,2010-12-01 08:26:00,0.001000,12346.000000,0.001000
25%,2.000000,2011-04-07 11:12:00,1.250000,13969.000000,4.680000
50%,6.000000,2011-07-31 14:39:00,1.950000,15159.000000,11.800000
75%,12.000000,2011-10-20 14:33:00,3.750000,16795.000000,19.800000
max,80995.000000,2011-12-09 12:50:00,8142.750000,18287.000000,168469.600000
std,179.331775,NaN,22.097877,1713.141560,309.071041


# Aplicación del RFM
El RFM es una técnica de segmentación de marketing que permite evaluar el valor de un cliente basándose en su comportamiento pasado. En lugar de mirar a toda tu base de datos como una "masa" uniforme, el RFM te dice quién es quién.